# Final HF Cohort Definition

This notebook's HF cohort definition is aligned with CreateHFCohort.sql's three inclusion rules used in JHHS EHR.

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import skew, kurtosis
from typing import List, Dict
from google.cloud import bigquery

client = bigquery.Client()
dataset = os.environ["WORKSPACE_CDR"]

## Cohort Definition

**Population**
Adults with an incident heart failure diagnosis, identified via the standardized concept hierarchy rooted at OMOP concept 316139 (Heart failure) and its descendants. Each patient's index date was set to their first recorded heart failure diagnosis date.

**Eligibility filters**
Restricted to patients whose sex at birth was recorded as female, aged 18–55 years as of the current date.

**Prior visit history**
At least one outpatient visit (OMOP concept 9202 and descendants) occurring ≥180 days before the index diagnosis date.

**Prior symptom/sign evidence**
At least one heart failure symptom or sign, drawn from a curated concept list (`feature_concepts_full`) and expanded to include all hierarchical descendants via `concept_ancestor`, occurring any time between the start of the observation period and strictly before the index diagnosis date.

**Corroborating diagnostic evidence**
Either an elevated natriuretic peptide measurement — NT-proBNP ≥125 pg/mL (concept 3029187 and descendants) or BNP ≥35 pg/mL (concept 3011960 and descendants) — or a pulmonary edema diagnosis (concept 4078925 and descendants, using condition era logic to collapse recurring records into discrete episodes), occurring between the start of the observation period and up to 7 days after the index diagnosis date.

**Final inclusion**
Patients satisfying all three evidentiary requirements — prior visit history, prior symptom evidence, and corroborating diagnostic evidence — form the final cohort. Demographic characteristics (age at diagnosis, race, ethnicity, sex at birth) and the index diagnosis date are retained for cohort characterization.

In [ ]:
feature_concepts_full = pd.read_csv("feature_concepts_full.csv") # HF-symptoms (used for HF cohort definition)

In [ ]:
symptom_concept_ids = feature_concepts_full['concept_id'].astype(int).tolist()

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter("symptom_concept_ids", "INT64", symptom_concept_ids),
    ]
)

final_cohort_sql = """
    WITH hf_diagnosis AS (
        SELECT DISTINCT 
            person_id,
            MIN(entry_datetime) as first_hf_diagnosis_date
        FROM `""" + dataset + """.cb_search_all_events` 
        WHERE 
            is_standard = 1
            AND concept_id IN (
                SELECT DISTINCT c.concept_id 
                FROM `""" + dataset + """.cb_criteria` c 
                JOIN (
                    SELECT CAST(cr.id as string) AS id       
                    FROM `""" + dataset + """.cb_criteria` cr       
                    WHERE concept_id IN (316139) 
                ) a 
                ON (
                    c.path LIKE CONCAT('%.', a.id, '.%') 
                    OR c.path LIKE CONCAT('%.', a.id) 
                    OR c.path LIKE CONCAT(a.id, '.%') 
                    OR c.path = a.id
                ) 
                WHERE is_standard = 1 AND is_selectable = 1
            )
            AND person_id IN (
                SELECT person_id 
                FROM `""" + dataset + """.person` p 
                WHERE sex_at_birth_concept_id IN (45878463)
            )
            AND person_id IN (
                SELECT person_id 
                FROM `""" + dataset + """.cb_search_person` p 
                WHERE
                    DATE_DIFF(CURRENT_DATE, dob, YEAR) - 
                    IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > 
                       EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) 
                    BETWEEN 18 AND 55 
            )
        GROUP BY person_id
    ),

    hf_diagnosis_with_op AS (
        SELECT
            hf.person_id,
            hf.first_hf_diagnosis_date,
            op.observation_period_start_date AS op_start_date,
            op.observation_period_end_date AS op_end_date
        FROM hf_diagnosis hf
        JOIN `""" + dataset + """.observation_period` op
            ON op.person_id = hf.person_id
            AND DATE(hf.first_hf_diagnosis_date) >= op.observation_period_start_date
            AND DATE(hf.first_hf_diagnosis_date) <= op.observation_period_end_date
    ),

    visit_history_evidence AS (
        SELECT DISTINCT hf.person_id
        FROM hf_diagnosis_with_op hf
        JOIN `""" + dataset + """.visit_occurrence` vo
            ON vo.person_id = hf.person_id
        WHERE
            vo.visit_concept_id IN (
                SELECT DISTINCT ca.descendant_concept_id
                FROM `""" + dataset + """.concept_ancestor` ca
                JOIN `""" + dataset + """.concept` c
                    ON c.concept_id = ca.descendant_concept_id
                WHERE ca.ancestor_concept_id = 9202
                    AND c.invalid_reason IS NULL
            )
            AND DATE(vo.visit_start_date) >= hf.op_start_date
            AND DATE(vo.visit_start_date) <= hf.op_end_date
            AND DATE(vo.visit_start_date) <= DATE_SUB(DATE(hf.first_hf_diagnosis_date), INTERVAL 180 DAY)
    ),

    ntprobnp_evidence AS (
        SELECT DISTINCT hf.person_id
        FROM hf_diagnosis_with_op hf
        JOIN `""" + dataset + """.measurement` m
            ON m.person_id = hf.person_id
        WHERE
            (
                -- NT-proBNP: threshold 125 pg/mL
                (
                    m.measurement_concept_id IN (
                        SELECT DISTINCT ca.descendant_concept_id
                        FROM `""" + dataset + """.concept_ancestor` ca
                        JOIN `""" + dataset + """.concept` c
                            ON c.concept_id = ca.descendant_concept_id
                        WHERE ca.ancestor_concept_id = 3029187
                            AND c.invalid_reason IS NULL
                    )
                    AND m.value_as_number >= 125.0
                )
                OR
                -- BNP: threshold 35 pg/mL
                (
                    m.measurement_concept_id IN (
                        SELECT DISTINCT ca.descendant_concept_id
                        FROM `""" + dataset + """.concept_ancestor` ca
                        JOIN `""" + dataset + """.concept` c
                            ON c.concept_id = ca.descendant_concept_id
                        WHERE ca.ancestor_concept_id = 3011960
                            AND c.invalid_reason IS NULL
                    )
                    AND m.value_as_number >= 35.0
                )
            )
            AND DATE(m.measurement_date) >= hf.op_start_date
            AND DATE(m.measurement_date) <= DATE_ADD(DATE(hf.first_hf_diagnosis_date), INTERVAL 7 DAY)
    ),

    pulmonary_edema_evidence AS (
        SELECT DISTINCT hf.person_id
        FROM hf_diagnosis_with_op hf
        JOIN `""" + dataset + """.condition_era` ce
            ON ce.person_id = hf.person_id
        WHERE
            ce.condition_concept_id IN (
                SELECT DISTINCT ca.descendant_concept_id
                FROM `""" + dataset + """.concept_ancestor` ca
                JOIN `""" + dataset + """.concept` c
                    ON c.concept_id = ca.descendant_concept_id
                WHERE ca.ancestor_concept_id = 4078925
                    AND c.invalid_reason IS NULL
            )
            AND DATE(ce.condition_era_start_date) >= hf.op_start_date
            AND DATE(ce.condition_era_start_date) <= DATE_ADD(DATE(hf.first_hf_diagnosis_date), INTERVAL 7 DAY)
    ),

    symptom_evidence AS (
        SELECT DISTINCT hf.person_id
        FROM hf_diagnosis_with_op hf
        JOIN `""" + dataset + """.condition_occurrence` co
            ON co.person_id = hf.person_id
        WHERE
            co.condition_concept_id IN (
                SELECT DISTINCT ca.descendant_concept_id
                FROM `""" + dataset + """.concept_ancestor` ca
                JOIN `""" + dataset + """.concept` c
                    ON c.concept_id = ca.descendant_concept_id
                WHERE ca.ancestor_concept_id IN UNNEST(@symptom_concept_ids)
                    AND c.invalid_reason IS NULL
            )
            AND DATE(co.condition_start_date) >= hf.op_start_date
            AND DATE(co.condition_start_date) < DATE(hf.first_hf_diagnosis_date)
    ),

    corroborating_evidence AS (
        SELECT person_id FROM ntprobnp_evidence
        UNION DISTINCT
        SELECT person_id FROM pulmonary_edema_evidence
    ),

    final_cohort_ids AS (
        SELECT hf.person_id, hf.first_hf_diagnosis_date
        FROM hf_diagnosis_with_op hf
        WHERE hf.person_id IN (SELECT person_id FROM corroborating_evidence)
        AND hf.person_id IN (SELECT person_id FROM visit_history_evidence)
        AND hf.person_id IN (SELECT person_id FROM symptom_evidence)
    )

    -- final output: cohort ids + demographic/diagnosis columns
    SELECT
        fc.person_id AS PERSON_ID,
        fc.first_hf_diagnosis_date,
        DATE_DIFF(
            DATE(fc.first_hf_diagnosis_date), sp.dob, YEAR
        ) - IF(
            EXTRACT(MONTH FROM sp.dob)*100 + EXTRACT(DAY FROM sp.dob) >
            EXTRACT(MONTH FROM DATE(fc.first_hf_diagnosis_date))*100 + EXTRACT(DAY FROM DATE(fc.first_hf_diagnosis_date)),
            1, 0
        ) AS age_at_diagnosis,
        sp.dob AS date_of_birth,
        p.sex_at_birth_concept_id,
        sac_sex.concept_name AS sex_at_birth,
        p.race_concept_id,
        sac_race.concept_name AS race,
        p.ethnicity_concept_id,
        sac_eth.concept_name AS ethnicity
    FROM final_cohort_ids fc
    JOIN `""" + dataset + """.person` p
        ON p.person_id = fc.person_id
    JOIN `""" + dataset + """.cb_search_person` sp
        ON sp.person_id = fc.person_id
    LEFT JOIN `""" + dataset + """.concept` sac_sex
        ON sac_sex.concept_id = p.sex_at_birth_concept_id
    LEFT JOIN `""" + dataset + """.concept` sac_race
        ON sac_race.concept_id = p.race_concept_id
    LEFT JOIN `""" + dataset + """.concept` sac_eth
        ON sac_eth.concept_id = p.ethnicity_concept_id
"""


final_cohort_df = client.query(final_cohort_sql, job_config=job_config).to_dataframe()
print(f"Final cohort size: {len(final_cohort_df):,}")

In [ ]:
# Save to CSV
final_cohort_df.to_csv("unique_patient_ids_with_hf_date_deaths.csv", index=False)
print("Saved to unique_patient_ids_with_hf_date_deaths.csv")

In [ ]:
# Summarize and visualize the year of first recorded HF diagnosis.
plot_df = final_cohort_df[
    ["PERSON_ID", "first_hf_diagnosis_date"]
].copy()

plot_df["first_hf_diagnosis_date"] = pd.to_datetime(
    plot_df["first_hf_diagnosis_date"], errors="raise"
)

# Retain one first diagnosis per patient.
plot_df = (
    plot_df.groupby("PERSON_ID", as_index=False)
    ["first_hf_diagnosis_date"].min()
)
plot_df["diagnosis_year"] = (
    plot_df["first_hf_diagnosis_date"].dt.year.astype("Int64")
)

valid_years = plot_df["diagnosis_year"].dropna().astype(int)
year_distribution = valid_years.value_counts().sort_index()

print("Summary Statistics:")
print(f"Total patients: {len(plot_df):,}")
print(f"Patients with missing diagnosis dates: {plot_df['diagnosis_year'].isna().sum():,}")

if not valid_years.empty:
    print(f"Earliest diagnosis year: {valid_years.min()}")
    print(f"Latest diagnosis year: {valid_years.max()}")
    print(f"Median diagnosis year: {valid_years.median():.1f}")

year_group_labels = [
    "Before 2010",
    "2010-2012",
    "2013-2015",
    "2016-2018",
    "2019-2021",
    "2022-2024",
    "2025 or later",
]

plot_df["year_group"] = pd.cut(
    plot_df["diagnosis_year"].astype(float),
    bins=[-np.inf, 2010, 2013, 2016, 2019, 2022, 2025, np.inf],
    labels=year_group_labels,
    right=False,
)

grouped_dist = (
    plot_df["year_group"]
    .value_counts(sort=False)
    .reindex(year_group_labels, fill_value=0)
)

print("\nPatients by diagnosis-year group:")
print(grouped_dist.to_string())

if not year_distribution.empty:
    year_distribution = year_distribution.reindex(
        range(valid_years.min(), valid_years.max() + 1),
        fill_value=0,
    )

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(
        year_distribution.index,
        year_distribution.values,
        color="steelblue",
    )

    ax.set_xlabel("Year of First Recorded HF Diagnosis", fontsize=12)
    ax.set_ylabel("Number of Patients", fontsize=12)
    ax.set_title(
        "HF Diagnosis Year Distribution",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xticks(year_distribution.index)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)

    if valid_years.min() <= 2016 <= valid_years.max():
        ax.axvline(
            x=2016,
            color="red",
            linestyle="--",
            linewidth=2,
            label="2016",
        )
        ax.legend()

    plt.tight_layout()
    plt.show()

    total_with_dates = len(valid_years)
    patients_2016_onwards = int((valid_years >= 2016).sum())
    percentage_2016_onwards = (
        patients_2016_onwards / total_with_dates * 100
    )

    print(f"\nPatients with valid diagnosis dates: {total_with_dates:,}")
    print(f"Patients diagnosed in 2016 or later: {patients_2016_onwards:,}")
    print(
        "Percentage diagnosed in 2016 or later "
        f"(among patients with valid dates): {percentage_2016_onwards:.2f}%"
    )
else:
    print("\nNo valid diagnosis dates available for visualization.")